# Diabetes Prediction — Corrected Analysis

This notebook performs exploratory data analysis (EDA), leakage-safe preprocessing,
model evaluation, and final model export.

**Important:** This is an educational exercise, not a clinical diagnostic system.
The final model is evaluated on a stratified holdout set. The exported model is then
refit on all available rows for deployment; its holdout metrics remain the evaluation
result from the separate training/test experiment.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, ConfusionMatrixDisplay,
    RocCurveDisplay
)

RANDOM_STATE = 42
DATA_PATH = "data/diabetes.csv"  # Change if running from another directory
df = pd.read_csv(DATA_PATH)
FEATURES = [
    "Pregnancies", "Glucose", "BloodPressure", "SkinThickness",
    "Insulin", "BMI", "DiabetesPedigreeFunction", "Age"
]
TARGET = "Outcome"
ZERO_AS_MISSING = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

df.head()

In [ ]:
print("Shape:", df.shape)
print("\\nData types and non-null counts:")
display(df.info())
print("\\nNull counts:")
display(df.isna().sum())
print("\\nDescriptive statistics:")
display(df.describe().T)
print("\\nTarget counts:")
display(df[TARGET].value_counts().rename(index={0: "Non-diabetic", 1: "Diabetic"}))

In [ ]:
# The dataset uses zero for some measurements where zero is not physiologically
# plausible. Report these counts before treating them as missing.
zero_counts = (df[ZERO_AS_MISSING] == 0).sum().sort_values(ascending=False)
display(zero_counts.rename("zero_count").to_frame())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=df, x=TARGET, ax=axes[0])
axes[0].set_title("Outcome class counts")
axes[0].set_xticklabels(["Non-diabetic (0)", "Diabetic (1)"])
sns.histplot(data=df, x="Glucose", hue=TARGET, kde=True, element="step",
             stat="density", common_norm=False, ax=axes[1])
axes[1].set_title("Glucose distribution by outcome")
plt.tight_layout()
plt.show()

In [ ]:
# Compare selected measurements by outcome (raw dataset values).
display(df.groupby(TARGET)[["Glucose", "BMI", "Age", "Pregnancies"]].mean().T)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(data=df, x=TARGET, y="Glucose", ax=axes[0])
axes[0].set_title("Glucose by outcome")
sns.boxplot(data=df, x=TARGET, y="BMI", ax=axes[1])
axes[1].set_title("BMI by outcome")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 7))
sns.heatmap(df.corr(numeric_only=True), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Feature correlation matrix")
plt.tight_layout()
plt.show()

In [ ]:
# Separate the untouched holdout set BEFORE any fitting or resampling.
X = df[FEATURES].copy()
y = df[TARGET].astype(int)

# Replace physiologically implausible zeros with NaN; imputation is learned
# inside the pipeline using training data only.
X[ZERO_AS_MISSING] = X[ZERO_AS_MISSING].replace(0, np.nan)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision (diabetic class)": precision_score(y_test, y_pred, zero_division=0),
    "Recall (diabetic class)": recall_score(y_test, y_pred, zero_division=0),
    "F1 (diabetic class)": f1_score(y_test, y_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, y_prob),
}
pd.Series(metrics, name="Holdout score").to_frame()

In [ ]:
print(classification_report(
    y_test, y_pred, target_names=["Non-diabetic", "Diabetic"], zero_division=0
))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=["Non-diabetic", "Diabetic"], cmap="Blues"
)
plt.title("Confusion matrix — stratified holdout set")
plt.tight_layout()
plt.show()

RocCurveDisplay.from_predictions(y_test, y_prob)
plt.title("ROC curve — stratified holdout set")
plt.tight_layout()
plt.show()

In [ ]:
# Optional: stratified 5-fold cross-validation on the development dataset.
# All imputation/model fitting occurs separately inside each fold.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_results = cross_validate(
    pipeline, X, y, cv=cv,
    scoring=["accuracy", "precision", "recall", "f1", "roc_auc"],
    n_jobs=-1
)
pd.DataFrame({
    metric: [cv_results[f"test_{metric}"].mean(), cv_results[f"test_{metric}"].std()]
    for metric in ["accuracy", "precision", "recall", "f1", "roc_auc"]
}, index=["mean", "std"]).T

In [ ]:
# Refit the same pipeline on all available rows for deployment.
# Do not report training-set scores as test performance.
final_model = pipeline.fit(X, y)
joblib.dump(final_model, "diabetes_model.pkl")
print("Saved deployment model to diabetes_model.pkl")

## Interpretation and limitations

- The target is `Outcome` (0 or 1); the other eight columns are model inputs.
- Zero values in Glucose, BloodPressure, SkinThickness, Insulin, and BMI are treated as missing and median-imputed.
- The split is stratified and occurs before preprocessing/model fitting, avoiding leakage from test data.
- `class_weight="balanced"` handles class imbalance without duplicating rows before the split.
- Report precision, recall, F1, ROC-AUC and the confusion matrix alongside accuracy.
- The Pima dataset is relatively small and has population-specific limitations. Results do not establish clinical usefulness or generalization.
- The final exported pipeline includes imputation and the classifier, so the app applies the same preprocessing automatically.